# 03 - ETL gobernado, cuarentena y carga atómica

Esta notebook acompaña la Unidad III y muestra cómo llevar las reglas de calidad a un flujo de **extracción, transformación y carga (ETL)**. El recorrido será: crear datos de práctica, normalizarlos, evaluarlos, decidir si se publican y registrar el resultado.

## Objetivos de aprendizaje

- Diferenciar los datos de entrada, las filas aceptadas, las rechazadas y los hallazgos.
- Explicar por qué un lote completo puede quedar bloqueado aunque algunas filas sean válidas.
- Reconocer qué información se registra para auditoría y qué contenido no se persiste en esta demostración.
- Observar controles de publicación, transacciones, rollback, idempotencia y reconciliación.

## Antes de comenzar

- Ejecuta las celdas **en orden, desde arriba hacia abajo**. Cada sección utiliza variables creadas en las anteriores.
- Se requiere Python y pandas. `sqlite3`, `decimal`, `hashlib`, `re` y `uuid` pertenecen a la biblioteca estándar.
- Los datos ficticios, las reglas y las funciones de transformación se definen en esta notebook.
- No necesitas otros archivos de código ni servicios externos. La base `:memory:` empieza vacía y no conserva los datos al cerrar la conexión.

> **Idea clave:** poner una fila en cuarentena deja evidencia para investigarla; no significa borrarla ni corregirla sin autorización.

**Dinámica de clase:** ejecuta una sola celda de código con `Shift+Enter`, observa su salida y comenta la pregunta antes de avanzar. Las funciones se presentan cuando se necesitan; no hay que esperar al final para ver los resultados. Para repetir toda la práctica, reinicia el kernel y vuelve al paso 1. No repitas solo una celda de carga: modifica el historial de intentos.


## 1. Preparar el entorno

**Acción:** importa las bibliotecas y fija el contrato y la fecha de corte. Solo pandas es una dependencia externa.

**Salida:** versión de pandas, fecha de corte y versión del contrato.

**Para conversar:** ¿por qué usamos una fecha fija y no la fecha actual?


In [26]:
import hashlib  # Calcula una huella para identificar cada lote de entrada.
import re  # Valida identificadores y formatos sencillos de texto.
import sqlite3  # Proporciona la base de datos local del laboratorio.
from decimal import Decimal, InvalidOperation  # Convierte importes con precisión decimal.
from uuid import uuid4  # Genera un identificador para cada intento de carga.

import pandas as pd  # Permite crear, transformar y analizar tablas.

FECHA_CORTE = pd.Timestamp("2026-10-06T12:00:00Z")  # Fija la referencia temporal del ejercicio.
VERSION_CONTRATO = "1.0.0"  # Identifica la versión de las reglas aplicadas.
COLUMNAS_FUENTE = {
    "fila_fuente", "id_venta", "fecha_venta", "id_producto", "monto",
    "moneda", "email", "actualizado_en",
}  # Declara las columnas obligatorias del archivo de entrada.

print('pandas:', pd.__version__)
print('Fecha de corte:', FECHA_CORTE)
print('Versión del contrato:', VERSION_CONTRATO)


pandas: 3.0.2
Fecha de corte: 2026-10-06 12:00:00+00:00
Versión del contrato: 1.0.0


## 2. Crear e inspeccionar los datos de entrada

**Acción:** crea las ventas ficticias, el maestro de productos y la evidencia parcial de exactitud. No se leen archivos externos.

**Salida:** las 12 filas crudas y las dos tablas de referencia.

**Para conversar:** identifica un ID repetido, una fecha imposible y una moneda fuera del dominio.


In [27]:
def crear_datos():
    """Retorna ventas ficticias, maestro y evidencia parcial independiente."""
    # Cada fila incluye un caso válido o un problema intencional para las reglas.
    filas = [
        ("v01", "1", "2026-10-05", "101", "1500.00", " ars ", "uno@demo.invalid", "2026-10-06T10:00:00Z"),
        ("v02", "2", "2026-10-05", "102", "200.00", "ARS", "dos@demo.invalid", "2026-10-06T10:00:00Z"),
        ("v03", "2", "2026-10-05", "102", "250.00", "ARS", "dos@demo.invalid", "2026-10-06T10:00:00Z"),
        ("v04", "3", "2026-02-29", "101", "300.00", "ARS", "tres@demo.invalid", "2026-10-06T10:00:00Z"),
        ("v05", "4", "2026-10-05", "101", "-10.00", "ARS", "cuatro@demo.invalid", "2026-10-06T10:00:00Z"),
        ("v06", "5", "2026-10-05", "101", "500.00", "BTC", "cinco@demo.invalid", "2026-10-06T10:00:00Z"),
        ("v07", "6", "2026-10-05", "999", "600.00", "ARS", "seis@demo.invalid", "2026-10-06T10:00:00Z"),
        ("v08", "7", "2026-10-05", "102", "700.00", "USD", "email-invalido", "2026-10-06T10:00:00Z"),
        ("v09", "8", "2026-09-30", "102", "800.00", "ARS", " ", "2026-10-01T10:00:00Z"),
        ("v10", "9", "2026-10-07", "101", "900.00", "ARS", None, "2026-10-06T10:00:00Z"),
        ("v11", "10", "2026-10-05", "101", "1000.00", "ARS", "diez@demo.invalid", "2026-10-06T10:00:00Z"),
        ("v12", " ", "2026-10-05", "101", "1200.00", "ARS", None, "2026-10-06T10:00:00Z"),
    ]
    # Conserva los valores originales como texto para evaluar su normalización.
    ventas = pd.DataFrame(filas, columns=[
        "fila_fuente", "id_venta", "fecha_venta", "id_producto", "monto",
        "moneda", "email", "actualizado_en",
    ])
    # Este maestro permite probar la integridad referencial de los productos.
    productos = pd.DataFrame({"id_producto": [101, 102], "producto": ["Producto A", "Producto B"]})
    referencia = pd.DataFrame({"id_venta": [1, 10], "monto_centavos_referencia": [150000, 99900]})
    return ventas, productos, referencia

crudos, productos, referencia = crear_datos()
print('Ventas de entrada:', len(crudos))
display(crudos)
print('Maestro de productos:')
display(productos)
print('Evidencia disponible para exactitud (importes en centavos):')
display(referencia)


Ventas de entrada: 12


,fila_fuente,id_venta,fecha_venta,id_producto,monto,moneda,email,actualizado_en
0,v01,1,2026-10-05,101,1500.00,ars,uno@demo.invalid,2026-10-06T10:00:00Z
1,v02,2,2026-10-05,102,200.00,ARS,dos@demo.invalid,2026-10-06T10:00:00Z
2,v03,2,2026-10-05,102,250.00,ARS,dos@demo.invalid,2026-10-06T10:00:00Z
3,v04,3,2026-02-29,101,300.00,ARS,tres@demo.invalid,2026-10-06T10:00:00Z
4,v05,4,2026-10-05,101,-10.00,ARS,cuatro@demo.invalid,2026-10-06T10:00:00Z
5,v06,5,2026-10-05,101,500.00,BTC,cinco@demo.invalid,2026-10-06T10:00:00Z
6,v07,6,2026-10-05,999,600.00,ARS,seis@demo.invalid,2026-10-06T10:00:00Z
7,v08,7,2026-10-05,102,700.00,USD,email-invalido,2026-10-06T10:00:00Z
8,v09,8,2026-09-30,102,800.00,ARS,,2026-10-01T10:00:00Z
9,v10,9,2026-10-07,101,900.00,ARS,NaN,2026-10-06T10:00:00Z


Maestro de productos:


,id_producto,producto
0,101,Producto A
1,102,Producto B


Evidencia disponible para exactitud (importes en centavos):


,id_venta,monto_centavos_referencia
0,1,150000
1,10,99900


## 3. Normalizar sin modificar la fuente

**Acción:** convierte identificadores, importes a centavos, fechas y formatos de texto sobre una copia. Una fecha imposible queda como `NaT`; no se inventa una corrección.

**Salida:** tabla normalizada y comparación con el formato original de las monedas.

**Para conversar:** ¿normalizar ` ars ` garantiza que todos los datos de la fila sean correctos?


In [28]:
def esquema_valido(datos):
    return set(datos.columns) == COLUMNAS_FUENTE


def centavos(valor):
    try:
        numero = Decimal(str(valor).strip())
        if not numero.is_finite() or numero != numero.quantize(Decimal("0.01")):
            return None
        return int(numero * 100)
    except (InvalidOperation, ValueError, TypeError):
        return None


def entero_positivo(valor):
    texto = str(valor).strip()
    return int(texto) if re.fullmatch(r"[1-9][0-9]*", texto) else None


def normalizar(datos):
    if not esquema_valido(datos):
        raise ValueError("Deriva de esquema: revisar contrato antes de transformar")
    if datos["fila_fuente"].isna().any() or datos["fila_fuente"].duplicated().any():
        raise ValueError("La identificacion tecnica de origen debe ser unica y no nula")
    # La copia permite comparar la entrada cruda con los datos transformados.
    resultado = datos.copy(deep=True)
    for columna in ["id_venta", "id_producto"]:
        resultado[columna] = pd.array(resultado[columna].map(entero_positivo), dtype="Int64")
    resultado["monto_centavos"] = pd.array(resultado["monto"].map(centavos), dtype="Int64")
    resultado["fecha_venta"] = pd.to_datetime(resultado["fecha_venta"], format="%Y-%m-%d", errors="coerce", utc=True)
    resultado["actualizado_en"] = pd.to_datetime(resultado["actualizado_en"], format="ISO8601", errors="coerce", utc=True)
    resultado["moneda"] = resultado["moneda"].astype("string").str.strip().str.upper()
    resultado["email"] = resultado["email"].astype("string").str.strip().replace("", pd.NA)
    # Devuelve la tabla normalizada y conserva intacta la entrada original.
    return resultado

fuente_antes = crudos.copy(deep=True)
normalizados = normalizar(crudos)
pd.testing.assert_frame_equal(crudos, fuente_antes)
display(normalizados)
display(pd.DataFrame({
    'fila_fuente': crudos['fila_fuente'],
    'moneda_original': crudos['moneda'],
    'moneda_normalizada': normalizados['moneda'],
}))
print('Fuente original conservada sin cambios.')


,fila_fuente,id_venta,fecha_venta,id_producto,monto,moneda,email,actualizado_en,monto_centavos
0,v01,1,2026-10-05 00:00:00+00:00,101,1500.00,ARS,uno@demo.invalid,2026-10-06 10:00:00+00:00,150000
1,v02,2,2026-10-05 00:00:00+00:00,102,200.00,ARS,dos@demo.invalid,2026-10-06 10:00:00+00:00,20000
2,v03,2,2026-10-05 00:00:00+00:00,102,250.00,ARS,dos@demo.invalid,2026-10-06 10:00:00+00:00,25000
3,v04,3,NaT,101,300.00,ARS,tres@demo.invalid,2026-10-06 10:00:00+00:00,30000
4,v05,4,2026-10-05 00:00:00+00:00,101,-10.00,ARS,cuatro@demo.invalid,2026-10-06 10:00:00+00:00,-1000
5,v06,5,2026-10-05 00:00:00+00:00,101,500.00,BTC,cinco@demo.invalid,2026-10-06 10:00:00+00:00,50000
6,v07,6,2026-10-05 00:00:00+00:00,999,600.00,ARS,seis@demo.invalid,2026-10-06 10:00:00+00:00,60000
7,v08,7,2026-10-05 00:00:00+00:00,102,700.00,USD,email-invalido,2026-10-06 10:00:00+00:00,70000
8,v09,8,2026-09-30 00:00:00+00:00,102,800.00,ARS,<NA>,2026-10-01 10:00:00+00:00,80000
9,v10,9,2026-10-07 00:00:00+00:00,101,900.00,ARS,<NA>,2026-10-06 10:00:00+00:00,90000


,fila_fuente,moneda_original,moneda_normalizada
0,v01,ars,ARS
1,v02,ARS,ARS
2,v03,ARS,ARS
3,v04,ARS,ARS
4,v05,ARS,ARS
5,v06,BTC,BTC
6,v07,ARS,ARS
7,v08,USD,USD
8,v09,ARS,ARS
9,v10,ARS,ARS


Fuente original conservada sin cambios.


## 4. Evaluar las reglas de calidad

**Acción:** aplica las reglas a todas las filas normalizadas. Las reglas de cuarentena rechazan filas; las advertencias dejan evidencia pero no las rechazan. La exactitud solo se evalúa si hay referencia.

**Salida:** métricas por regla y detalle de los hallazgos.

**Para conversar:** ¿por qué una fila puede producir varios hallazgos?


In [29]:
def evaluar(datos, productos, referencia):
    """Evalua todas las filas; exactitud solo sobre evidencia disponible."""
    referencia = referencia.copy()
    if referencia["id_venta"].duplicated().any() or referencia["id_venta"].isna().any():
        raise ValueError("La referencia de exactitud tiene claves invalidas")
    if productos["id_producto"].duplicated().any() or productos["id_producto"].isna().any():
        raise ValueError("El maestro de productos tiene claves invalidas")
    verdad = datos["id_venta"].map(referencia.set_index("id_venta")["monto_centavos_referencia"])
    edad_horas = (FECHA_CORTE - datos["actualizado_en"]).dt.total_seconds() / 3600
    # Las reglas de cuarentena bloquean filas; las advertencias solo dejan evidencia.
    reglas = [
        ("DQ_ID", "completitud", "cuarentena", datos["id_venta"].notna(), None),
        ("DQ_UNICIDAD", "unicidad", "cuarentena", ~datos["id_venta"].duplicated(keep=False) & datos["id_venta"].notna(), None),
        ("DQ_FECHA", "validez", "cuarentena", datos["fecha_venta"].notna() & datos["fecha_venta"].le(FECHA_CORTE), None),
        ("DQ_MONTO", "validez", "cuarentena", datos["monto_centavos"].notna() & datos["monto_centavos"].gt(0), None),
        ("DQ_MONEDA", "integridad", "cuarentena", datos["moneda"].isin(["ARS", "USD", "EUR"]), None),
        ("DQ_FK_PRODUCTO", "integridad", "cuarentena", datos["id_producto"].isin(productos["id_producto"]), None),
        ("DQ_ACTUALIZACION", "consistencia", "cuarentena", datos["actualizado_en"].notna() & datos["actualizado_en"].le(FECHA_CORTE) & datos["actualizado_en"].ge(datos["fecha_venta"]), None),
        ("DQ_EMAIL", "validez", "advertencia", datos["email"].str.fullmatch(r"[^\s@]+@[^\s@]+\.[^\s@]+").fillna(False), None),
        ("DQ_VIGENCIA", "vigencia", "advertencia", edad_horas.between(0, 48), None),
        ("DQ_EXACTITUD", "exactitud", "advertencia", datos["monto_centavos"].eq(verdad), verdad.notna()),
    ]
    # Separa el detalle por fila del resumen agregado de cada regla.
    detalle, resumen = [], []
    for regla_id, dimension, accion, cumple, elegible in reglas:
        elegible = pd.Series(True, index=datos.index) if elegible is None else elegible
        cumple = cumple.fillna(False)
        falla = elegible & ~cumple
        evaluados = int(elegible.sum())
        fallidos = int(falla.sum())
        resumen.append({
            "regla": regla_id, "dimension": dimension, "accion": accion,
            "evaluados": evaluados, "fallidos": fallidos,
            "no_evaluables": len(datos) - evaluados,
            "cumplimiento_pct": 100 * (evaluados - fallidos) / evaluados if evaluados else None,
            "owner": "Gerencia Comercial", "contrato": VERSION_CONTRATO,
        })
        for fila_fuente in datos.loc[falla, "fila_fuente"]:
            detalle.append({"fila_fuente": fila_fuente, "regla": regla_id, "accion": accion})
    detalle = pd.DataFrame(detalle, columns=["fila_fuente", "regla", "accion"])
    # Una fila se rechaza si incumple al menos una regla marcada como cuarentena.
    ids_rechazados = set(detalle.loc[detalle["accion"].eq("cuarentena"), "fila_fuente"])
    rechazadas = datos[datos["fila_fuente"].isin(ids_rechazados)].copy()
    aceptadas = datos[~datos["fila_fuente"].isin(ids_rechazados)].copy()
    assert len(datos) == len(aceptadas) + len(rechazadas)
    return aceptadas, rechazadas, pd.DataFrame(resumen), detalle

aceptadas, rechazadas, metricas, detalle = evaluar(normalizados, productos, referencia)
print('Métricas por regla:')
display(metricas)
print('Hallazgos por fila y regla:')
display(detalle)


Métricas por regla:


,regla,dimension,accion,evaluados,fallidos,no_evaluables,cumplimiento_pct,owner,contrato
0,DQ_ID,completitud,cuarentena,12,1,0,91.666667,Gerencia Comercial,1.0.0
1,DQ_UNICIDAD,unicidad,cuarentena,12,3,0,75.000000,Gerencia Comercial,1.0.0
2,DQ_FECHA,validez,cuarentena,12,2,0,83.333333,Gerencia Comercial,1.0.0
3,DQ_MONTO,validez,cuarentena,12,1,0,91.666667,Gerencia Comercial,1.0.0
4,DQ_MONEDA,integridad,cuarentena,12,1,0,91.666667,Gerencia Comercial,1.0.0
5,DQ_FK_PRODUCTO,integridad,cuarentena,12,1,0,91.666667,Gerencia Comercial,1.0.0
6,DQ_ACTUALIZACION,consistencia,cuarentena,12,2,0,83.333333,Gerencia Comercial,1.0.0
7,DQ_EMAIL,validez,advertencia,12,4,0,66.666667,Gerencia Comercial,1.0.0
8,DQ_VIGENCIA,vigencia,advertencia,12,1,0,91.666667,Gerencia Comercial,1.0.0
9,DQ_EXACTITUD,exactitud,advertencia,2,1,10,50.000000,Gerencia Comercial,1.0.0


Hallazgos por fila y regla:


,fila_fuente,regla,accion
0,v12,DQ_ID,cuarentena
1,v02,DQ_UNICIDAD,cuarentena
2,v03,DQ_UNICIDAD,cuarentena
3,v12,DQ_UNICIDAD,cuarentena
4,v04,DQ_FECHA,cuarentena
5,v10,DQ_FECHA,cuarentena
6,v05,DQ_MONTO,cuarentena
7,v06,DQ_MONEDA,cuarentena
8,v07,DQ_FK_PRODUCTO,cuarentena
9,v04,DQ_ACTUALIZACION,cuarentena


## 5. Observar aceptadas y cuarentena

**Acción:** inspecciona por separado las filas aptas y las rechazadas. Todavía no se publica nada.

**Salida esperada:** 4 aceptadas y 8 rechazadas; juntas conservan las 12 filas de origen.

**Para conversar:** busca una fila aceptada con advertencias. La cuarentena de este ejercicio está en memoria; no es un depósito durable.


In [30]:
assert len(aceptadas) == 4 and len(rechazadas) == 8
assert len(crudos) == len(aceptadas) + len(rechazadas)
print('Filas aceptadas:', len(aceptadas))
display(aceptadas)
print('Filas en cuarentena:', len(rechazadas))
display(rechazadas)
print('No se eliminó ni se corrigió ninguna fila de origen.')


Filas aceptadas: 4


,fila_fuente,id_venta,fecha_venta,id_producto,monto,moneda,email,actualizado_en,monto_centavos
0,v01,1,2026-10-05 00:00:00+00:00,101,1500.00,ARS,uno@demo.invalid,2026-10-06 10:00:00+00:00,150000
7,v08,7,2026-10-05 00:00:00+00:00,102,700.00,USD,email-invalido,2026-10-06 10:00:00+00:00,70000
8,v09,8,2026-09-30 00:00:00+00:00,102,800.00,ARS,<NA>,2026-10-01 10:00:00+00:00,80000
10,v11,10,2026-10-05 00:00:00+00:00,101,1000.00,ARS,diez@demo.invalid,2026-10-06 10:00:00+00:00,100000


Filas en cuarentena: 8


,fila_fuente,id_venta,fecha_venta,id_producto,monto,moneda,email,actualizado_en,monto_centavos
1,v02,2,2026-10-05 00:00:00+00:00,102,200.00,ARS,dos@demo.invalid,2026-10-06 10:00:00+00:00,20000
2,v03,2,2026-10-05 00:00:00+00:00,102,250.00,ARS,dos@demo.invalid,2026-10-06 10:00:00+00:00,25000
3,v04,3,NaT,101,300.00,ARS,tres@demo.invalid,2026-10-06 10:00:00+00:00,30000
4,v05,4,2026-10-05 00:00:00+00:00,101,-10.00,ARS,cuatro@demo.invalid,2026-10-06 10:00:00+00:00,-1000
5,v06,5,2026-10-05 00:00:00+00:00,101,500.00,BTC,cinco@demo.invalid,2026-10-06 10:00:00+00:00,50000
6,v07,6,2026-10-05 00:00:00+00:00,999,600.00,ARS,seis@demo.invalid,2026-10-06 10:00:00+00:00,60000
9,v10,9,2026-10-07 00:00:00+00:00,101,900.00,ARS,<NA>,2026-10-06 10:00:00+00:00,90000
11,v12,<NA>,2026-10-05 00:00:00+00:00,101,1200.00,ARS,<NA>,2026-10-06 10:00:00+00:00,120000


No se eliminó ni se corrigió ninguna fila de origen.


## 6. Decidir si el lote puede publicarse

**Acción:** calcula el porcentaje de rechazo y aplica la puerta de publicación. Una huella identifica el contenido de entrada; no sustituye su conservación.

**Salida esperada:** rechazo del 66,7 %, por encima del máximo del 20 %; decisión `BLOQUEADO`.

**Para conversar:** ¿por qué no publicamos las 4 aceptadas como si el lote completo estuviera aprobado?


In [31]:
def puerta_publicacion(total, rechazadas, max_rechazo=0.20):
    # Valida los parámetros antes de calcular el porcentaje de rechazo.
    if not 0 <= max_rechazo <= 1 or total < 0 or not 0 <= rechazadas <= total:
        raise ValueError('Parámetros inválidos para la puerta de publicación')

    # No publica lotes vacíos, totalmente rechazados ni por encima del umbral.
    return total > 0 and rechazadas < total and rechazadas / total <= max_rechazo


def huella(datos):
    # Serializa las columnas y filas de forma estable para identificar la entrada.
    contenido = datos.to_json(orient='split', date_format='iso', index=False)
    return hashlib.sha256(contenido.encode('utf-8')).hexdigest()

porcentaje_rechazo = len(rechazadas) / len(crudos)
puede_publicarse = puerta_publicacion(len(crudos), len(rechazadas))
assert not puede_publicarse
print(f'Rechazo del lote: {porcentaje_rechazo:.1%} | Máximo permitido: 20%')
print('Decisión:', 'PUBLICADO' if puede_publicarse else 'BLOQUEADO')
print('Huella SHA-256:', huella(crudos))


Rechazo del lote: 66.7% | Máximo permitido: 20%
Decisión: BLOQUEADO
Huella SHA-256: 99d6adeb034a0308cb980ed47833d2fcb4d29946bd79afa2b387fcb3a23954f3


## 7. Crear e inspeccionar el destino SQLite

**Acción:** abre una base temporal y crea `productos`, `ventas`, `lotes` y `hallazgos`. Activa las claves foráneas.

**Salida:** tablas creadas, estructura de ventas y conteo inicial de 0 ventas.

**Para conversar:** identifica la clave primaria, los campos obligatorios y dónde se controla la moneda. La base desaparece cuando se cierra la conexión.


In [32]:
def nueva_conexion():
    # Crea una base en memoria que se reinicia en cada ejecución del laboratorio.
    conexion = sqlite3.connect(':memory:')
    conexion.execute('PRAGMA foreign_keys = ON')  # Activa la validación de claves foráneas.

    # Define el maestro de productos y la tabla de ventas con restricciones básicas.
    conexion.executescript('''
        CREATE TABLE productos (
            id_producto INTEGER PRIMARY KEY,
            producto TEXT NOT NULL
        );
        CREATE TABLE ventas (
            id_venta INTEGER PRIMARY KEY,
            id_producto INTEGER NOT NULL REFERENCES productos(id_producto),
            fecha_venta TEXT NOT NULL,
            monto_centavos INTEGER NOT NULL CHECK (monto_centavos > 0),
            moneda TEXT NOT NULL CHECK (moneda IN ('ARS', 'USD', 'EUR')),
            fila_fuente TEXT NOT NULL,
            lote_inicial TEXT NOT NULL
        );
        CREATE TABLE lotes (
            lote TEXT PRIMARY KEY,
            huella TEXT NOT NULL,
            contrato TEXT NOT NULL,
            estado TEXT NOT NULL,
            total INTEGER NOT NULL,
            aceptadas INTEGER NOT NULL,
            rechazadas INTEGER NOT NULL,
            nuevas INTEGER NOT NULL,
            existentes INTEGER NOT NULL
        );
        CREATE TABLE hallazgos (
            lote TEXT NOT NULL REFERENCES lotes(lote),
            fila_fuente TEXT NOT NULL,
            regla TEXT NOT NULL,
            accion TEXT NOT NULL
        );
    ''')
    return conexion

conexion = nueva_conexion()
assert conexion.execute('PRAGMA foreign_keys').fetchone()[0] == 1
print('Tablas creadas:')
display(pd.read_sql_query(
    "SELECT name AS tabla FROM sqlite_master WHERE type='table' ORDER BY name", conexion
))
print('Estructura de ventas:')
display(pd.read_sql_query('PRAGMA table_info(ventas)', conexion))
print('Ventas iniciales:', conexion.execute('SELECT COUNT(*) FROM ventas').fetchone()[0])


Tablas creadas:


,tabla
0,hallazgos
1,lotes
2,productos
3,ventas


Estructura de ventas:


,cid,name,type,notnull,dflt_value,pk
0,0,id_venta,INTEGER,0,None,1
1,1,id_producto,INTEGER,1,None,0
2,2,fecha_venta,TEXT,1,None,0
3,3,monto_centavos,INTEGER,1,None,0
4,4,moneda,TEXT,1,None,0
5,5,fila_fuente,TEXT,1,None,0
6,6,lote_inicial,TEXT,1,None,0


Ventas iniciales: 0


## 8. Registrar el lote bloqueado, sin publicar ventas

**Acción:** define el coordinador `ejecutar_etl` y úsalo con el lote original. Reutiliza las funciones que ya examinamos: normaliza, evalúa, decide y registra el intento.

**Salida esperada:** lote `BLOQUEADO`, 0 ventas y hallazgos registrados.

**Para conversar:** diferencia las filas aceptadas por las reglas de las ventas efectivamente publicadas. La función coordina la transacción; las demostraciones siguientes se ejecutan en celdas separadas.


In [33]:
def ejecutar_etl(datos_crudos, productos, referencia, conexion):
    # Normaliza la entrada y calcula resultados de calidad antes de escribir.
    datos = normalizar(datos_crudos)
    aceptadas, rechazadas, metricas, detalle = evaluar(datos, productos, referencia)

    # Asigna un ID al intento y decide si el lote completo puede publicarse.
    lote = uuid4().hex
    estado = 'PUBLICADO' if puerta_publicacion(len(datos), len(rechazadas)) else 'BLOQUEADO'
    nuevas, existentes = 0, 0  # Cuenta inserciones y recargas idénticas.

    try:
        # Toda publicación ocurre en una transacción: un error revierte sus escrituras.
        with conexion:
            if estado == 'PUBLICADO':
                # Agrega productos faltantes sin sobrescribir cambios del maestro.
                for producto in productos.itertuples(index=False):
                    existente = conexion.execute(
                        'SELECT producto FROM productos WHERE id_producto=?',
                        (int(producto.id_producto),),
                    ).fetchone()
                    if existente is None:
                        conexion.execute(
                            'INSERT INTO productos VALUES (?, ?)',
                            (int(producto.id_producto), producto.producto),
                        )
                    elif existente[0] != producto.producto:
                        raise ValueError('Cambio de maestro: requiere política SCD aprobada')

                # Inserta las ventas nuevas y detecta contenido conflictivo o repetido.
                for venta in aceptadas.itertuples(index=False):
                    valores = (
                        int(venta.id_producto),
                        venta.fecha_venta.date().isoformat(),
                        int(venta.monto_centavos),
                        str(venta.moneda),
                    )
                    existente = conexion.execute(
                        'SELECT id_producto, fecha_venta, monto_centavos, moneda '
                        'FROM ventas WHERE id_venta=?',
                        (int(venta.id_venta),),
                    ).fetchone()
                    if existente is None:
                        conexion.execute(
                            'INSERT INTO ventas VALUES (?, ?, ?, ?, ?, ?, ?)',
                            (int(venta.id_venta), *valores, venta.fila_fuente, lote),
                        )
                        nuevas += 1
                    elif existente == valores:
                        existentes += 1
                    else:
                        raise ValueError('Conflicto de contenido para una venta ya publicada')

                # Relee el destino y compara filas e importes, agrupados por moneda.
                claves = [int(identificador) for identificador in aceptadas['id_venta']]
                marcadores = ','.join('?' for _ in claves)
                destino = pd.read_sql_query(
                    f'SELECT id_venta, moneda, monto_centavos FROM ventas '
                    f'WHERE id_venta IN ({marcadores})',
                    conexion,
                    params=claves,
                )
                assert len(destino) == len(aceptadas)
                esperado = {
                    str(moneda): int(total)
                    for moneda, total in aceptadas.groupby('moneda')['monto_centavos'].sum().items()
                }
                obtenido = {
                    str(moneda): int(total)
                    for moneda, total in destino.groupby('moneda')['monto_centavos'].sum().items()
                }
                assert esperado == obtenido  # Confirma que los totales por moneda coincidan.

            # Registra cada intento, incluso si la política bloqueó el lote.
            conexion.execute(
                'INSERT INTO lotes VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)',
                (
                    lote,
                    huella(datos_crudos),
                    VERSION_CONTRATO,
                    estado,
                    len(datos),
                    len(aceptadas),
                    len(rechazadas),
                    nuevas,
                    existentes,
                ),
            )

            # Persiste metadatos de hallazgos; no guarda aquí las filas completas.
            conexion.executemany(
                'INSERT INTO hallazgos VALUES (?, ?, ?, ?)',
                [
                    (lote, fila.fila_fuente, fila.regla, fila.accion)
                    for fila in detalle.itertuples(index=False)
                ],
            )
    except Exception:
        # Si falla la publicación, registra el intento después del rollback y propaga el error.
        with conexion:
            conexion.execute(
                'INSERT INTO lotes VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)',
                (
                    lote,
                    huella(datos_crudos),
                    VERSION_CONTRATO,
                    'ERROR_TECNICO',
                    len(datos),
                    len(aceptadas),
                    len(rechazadas),
                    0,
                    0,
                ),
            )
        raise

    # Devuelve resultados tabulares para inspección y explicación en la clase.
    return {
        'lote': lote,
        'estado': estado,
        'nuevas': nuevas,
        'existentes': existentes,
        'aceptadas': aceptadas,
        'rechazadas': rechazadas,
        'metricas': metricas,
        'detalle': detalle,
    }

bloqueado = ejecutar_etl(crudos, productos, referencia, conexion)
assert bloqueado['estado'] == 'BLOQUEADO'
assert conexion.execute('SELECT COUNT(*) FROM ventas').fetchone()[0] == 0
print('Resultado:', bloqueado['estado'], '| Ventas publicadas: 0')
display(pd.read_sql_query(
    'SELECT * FROM lotes', conexion
))
print('Evidencia de los hallazgos:')
display(pd.read_sql_query('SELECT * FROM hallazgos', conexion))


Resultado: BLOQUEADO | Ventas publicadas: 0


,lote,huella,contrato,estado,total,aceptadas,rechazadas,nuevas,existentes
0,a8fe2f4dbc234bffaf568d4eda787bc3,99d6adeb034a0308cb980ed47833d2fcb4d29946bd79af...,1.0.0,BLOQUEADO,12,4,8,0,0


Evidencia de los hallazgos:


,lote,fila_fuente,regla,accion
0,a8fe2f4dbc234bffaf568d4eda787bc3,v12,DQ_ID,cuarentena
1,a8fe2f4dbc234bffaf568d4eda787bc3,v02,DQ_UNICIDAD,cuarentena
2,a8fe2f4dbc234bffaf568d4eda787bc3,v03,DQ_UNICIDAD,cuarentena
3,a8fe2f4dbc234bffaf568d4eda787bc3,v12,DQ_UNICIDAD,cuarentena
4,a8fe2f4dbc234bffaf568d4eda787bc3,v04,DQ_FECHA,cuarentena
5,a8fe2f4dbc234bffaf568d4eda787bc3,v10,DQ_FECHA,cuarentena
6,a8fe2f4dbc234bffaf568d4eda787bc3,v05,DQ_MONTO,cuarentena
7,a8fe2f4dbc234bffaf568d4eda787bc3,v06,DQ_MONEDA,cuarentena
8,a8fe2f4dbc234bffaf568d4eda787bc3,v07,DQ_FK_PRODUCTO,cuarentena
9,a8fe2f4dbc234bffaf568d4eda787bc3,v04,DQ_ACTUALIZACION,cuarentena


In [34]:
print('Resultado:', bloqueado['estado'], '| Ventas publicadas: 0')
display(pd.read_sql_query(
    'SELECT * FROM lotes', conexion
))
print('Evidencia de los hallazgos:')
display(pd.read_sql_query('SELECT * FROM hallazgos', conexion))

Resultado: BLOQUEADO | Ventas publicadas: 0


,lote,huella,contrato,estado,total,aceptadas,rechazadas,nuevas,existentes
0,a8fe2f4dbc234bffaf568d4eda787bc3,99d6adeb034a0308cb980ed47833d2fcb4d29946bd79af...,1.0.0,BLOQUEADO,12,4,8,0,0


Evidencia de los hallazgos:


,lote,fila_fuente,regla,accion
0,a8fe2f4dbc234bffaf568d4eda787bc3,v12,DQ_ID,cuarentena
1,a8fe2f4dbc234bffaf568d4eda787bc3,v02,DQ_UNICIDAD,cuarentena
2,a8fe2f4dbc234bffaf568d4eda787bc3,v03,DQ_UNICIDAD,cuarentena
3,a8fe2f4dbc234bffaf568d4eda787bc3,v12,DQ_UNICIDAD,cuarentena
4,a8fe2f4dbc234bffaf568d4eda787bc3,v04,DQ_FECHA,cuarentena
5,a8fe2f4dbc234bffaf568d4eda787bc3,v10,DQ_FECHA,cuarentena
6,a8fe2f4dbc234bffaf568d4eda787bc3,v05,DQ_MONTO,cuarentena
7,a8fe2f4dbc234bffaf568d4eda787bc3,v06,DQ_MONEDA,cuarentena
8,a8fe2f4dbc234bffaf568d4eda787bc3,v07,DQ_FK_PRODUCTO,cuarentena
9,a8fe2f4dbc234bffaf568d4eda787bc3,v04,DQ_ACTUALIZACION,cuarentena


## 9. Preparar un subconjunto de prueba apto

**Acción:** selecciona de la fuente las 4 filas aceptadas y muéstralas antes de cargar. No se escribe todavía en SQLite.

**Salida esperada:** 4 filas y 0 ventas en destino.

**Para conversar:** esto no repara el lote original. Las 8 rechazadas siguen pendientes de investigación; publicar un subconjunto en producción requiere alcance y autorización explícitos.


In [35]:
ids_aptos = set(bloqueado['aceptadas']['fila_fuente'])
subconjunto = crudos[crudos['fila_fuente'].isin(ids_aptos)].copy()
assert len(subconjunto) == 4
print('Subconjunto de prueba; filas:', len(subconjunto))
display(subconjunto)
assert conexion.execute('SELECT COUNT(*) FROM ventas').fetchone()[0] == 0
print('Ventas en destino antes de cargar: 0. Lote original todavía bloqueado.')


Subconjunto de prueba; filas: 4


,fila_fuente,id_venta,fecha_venta,id_producto,monto,moneda,email,actualizado_en
0,v01,1,2026-10-05,101,1500.00,ars,uno@demo.invalid,2026-10-06T10:00:00Z
7,v08,7,2026-10-05,102,700.00,USD,email-invalido,2026-10-06T10:00:00Z
8,v09,8,2026-09-30,102,800.00,ARS,,2026-10-01T10:00:00Z
10,v11,10,2026-10-05,101,1000.00,ARS,diez@demo.invalid,2026-10-06T10:00:00Z


Ventas en destino antes de cargar: 0. Lote original todavía bloqueado.


## 10. Publicar el subconjunto de prueba

**Acción:** carga el subconjunto en una transacción y observa las ventas resultantes.

**Salida esperada:** estado `PUBLICADO`, 4 ventas nuevas y ninguna columna de email en el destino.

**Para conversar:** ¿qué información de trazabilidad relaciona una venta con su fila de origen y su primer lote?


In [36]:
publicado = ejecutar_etl(subconjunto, productos, referencia, conexion)
assert publicado['estado'] == 'PUBLICADO' and publicado['nuevas'] == 4
ventas_publicadas = pd.read_sql_query('SELECT * FROM ventas ORDER BY id_venta', conexion)
assert len(ventas_publicadas) == 4
assert 'email' not in ventas_publicadas.columns
print('Estado:', publicado['estado'], '| Nuevas:', publicado['nuevas'])
display(ventas_publicadas)
print('El destino analítico no incluye email; el lote original permanece bloqueado.')


Estado: PUBLICADO | Nuevas: 4


,id_venta,id_producto,fecha_venta,monto_centavos,moneda,fila_fuente,lote_inicial
0,1,101,2026-10-05,150000,ARS,v01,3e036ddf70094df482ea85783799ec17
1,7,102,2026-10-05,70000,USD,v08,3e036ddf70094df482ea85783799ec17
2,8,102,2026-09-30,80000,ARS,v09,3e036ddf70094df482ea85783799ec17
3,10,101,2026-10-05,100000,ARS,v11,3e036ddf70094df482ea85783799ec17


El destino analítico no incluye email; el lote original permanece bloqueado.


## 11. Reconciliar cantidades e importes por moneda

**Acción:** compara los totales de las filas aceptadas del subconjunto con lo realmente cargado. El coordinador ya hizo esta comprobación dentro de la transacción; ahora la mostramos para explicarla.

**Salida esperada:** 4 filas en ambos lados; ARS = 330000 centavos y USD = 70000 centavos, con diferencias de 0.

**Para conversar:** no se suman ARS y USD en un único total sin una conversión aprobada.


In [37]:
esperado_por_moneda = publicado['aceptadas'].groupby('moneda')['monto_centavos'].sum()
obtenido_por_moneda = pd.read_sql_query(
    'SELECT moneda, SUM(monto_centavos) AS total_centavos FROM ventas GROUP BY moneda', conexion
).set_index('moneda')['total_centavos']
reconciliacion = pd.DataFrame({
    'esperado_centavos': esperado_por_moneda,
    'obtenido_centavos': obtenido_por_moneda,
})
reconciliacion['diferencia_centavos'] = (
    reconciliacion['obtenido_centavos'] - reconciliacion['esperado_centavos']
)
assert reconciliacion.notna().all().all()
assert reconciliacion['diferencia_centavos'].eq(0).all()
assert len(ventas_publicadas) == len(publicado['aceptadas'])
print('Filas esperadas:', len(publicado['aceptadas']), '| Filas en destino:', len(ventas_publicadas))
display(reconciliacion.reset_index())


Filas esperadas: 4 | Filas en destino: 4


,moneda,esperado_centavos,obtenido_centavos,diferencia_centavos
0,ARS,330000,330000,0
1,USD,70000,70000,0


## 12. Recargar sin duplicar: idempotencia

**Acción:** procesa otra vez el mismo subconjunto y compara el destino antes y después.

**Salida esperada:** 0 nuevas, 4 existentes y contenido idéntico. La auditoría registra otro intento.

**Para conversar:** idempotencia del dato no significa borrar el historial de ejecuciones.


In [38]:
antes = pd.read_sql_query('SELECT * FROM ventas ORDER BY id_venta', conexion)
recarga = ejecutar_etl(subconjunto, productos, referencia, conexion)
despues_recarga = pd.read_sql_query('SELECT * FROM ventas ORDER BY id_venta', conexion)
assert recarga['nuevas'] == 0 and recarga['existentes'] == 4
pd.testing.assert_frame_equal(antes, despues_recarga)
print('Nuevas:', recarga['nuevas'], '| Existentes:', recarga['existentes'])
display(despues_recarga)
print('Intentos registrados, incluida la recarga:')
display(pd.read_sql_query(
    'SELECT estado, total, nuevas, existentes FROM lotes ORDER BY rowid', conexion
))


Nuevas: 0 | Existentes: 4


,id_venta,id_producto,fecha_venta,monto_centavos,moneda,fila_fuente,lote_inicial
0,1,101,2026-10-05,150000,ARS,v01,3e036ddf70094df482ea85783799ec17
1,7,102,2026-10-05,70000,USD,v08,3e036ddf70094df482ea85783799ec17
2,8,102,2026-09-30,80000,ARS,v09,3e036ddf70094df482ea85783799ec17
3,10,101,2026-10-05,100000,ARS,v11,3e036ddf70094df482ea85783799ec17


Intentos registrados, incluida la recarga:


,estado,total,nuevas,existentes
0,BLOQUEADO,12,0,0
1,PUBLICADO,4,4,0
2,PUBLICADO,4,0,4


## 13. Preparar un conflicto antes de ejecutarlo

**Acción:** construye un lote de dos filas: primero una venta nueva con ID 20, luego una venta ya publicada con el monto alterado. No lo cargues hasta el siguiente paso.

**Salida:** lote conflictivo y venta original para comparar.

**Para conversar:** predice qué debería pasar con la venta nueva si la segunda fila genera un conflicto.


In [39]:
nueva = subconjunto.iloc[[0]].copy()
nueva.loc[:, 'id_venta'] = '20'
nueva.loc[:, 'fila_fuente'] = 'prueba-nueva'
conflicto = subconjunto.iloc[[0]].copy()
conflicto.loc[:, 'monto'] = '1501.00'
lote_conflictivo = pd.concat([nueva, conflicto], ignore_index=True)
print('Lote de prueba: primero venta nueva; después contenido conflictivo.')
display(lote_conflictivo)
print('Venta original publicada (importe expresado en centavos):')
display(antes[antes['id_venta'].eq(int(conflicto.iloc[0]['id_venta']))])
print('Todavía no se ha ejecutado este lote.')


Lote de prueba: primero venta nueva; después contenido conflictivo.


,fila_fuente,id_venta,fecha_venta,id_producto,monto,moneda,email,actualizado_en
0,prueba-nueva,20,2026-10-05,101,1500.00,ars,uno@demo.invalid,2026-10-06T10:00:00Z
1,v01,1,2026-10-05,101,1501.00,ars,uno@demo.invalid,2026-10-06T10:00:00Z


Venta original publicada (importe expresado en centavos):


,id_venta,id_producto,fecha_venta,monto_centavos,moneda,fila_fuente,lote_inicial
0,1,101,2026-10-05,150000,ARS,v01,3e036ddf70094df482ea85783799ec17


Todavía no se ha ejecutado este lote.


## 14. Ejecutar el conflicto y comprobar el rollback

**Acción:** intenta cargar el lote conflictivo y comprueba que se revierte completo.

**Salida esperada:** conflicto detectado, venta 20 ausente y las 4 ventas anteriores intactas.

**Para conversar:** el esquema didáctico registra el intento como `ERROR_TECNICO`, aunque este es un error de contenido. En producción hay que distinguirlo de fallas transitorias antes de decidir reintentos.


In [40]:
fallo_detectado = False
try:
    ejecutar_etl(lote_conflictivo, productos, referencia, conexion)
except ValueError as error:
    if 'Conflicto de contenido' not in str(error):
        raise
    fallo_detectado = True
    print('Conflicto esperado:', error)
assert fallo_detectado
ventas_tras_rollback = pd.read_sql_query('SELECT * FROM ventas ORDER BY id_venta', conexion)
venta_20 = pd.read_sql_query('SELECT * FROM ventas WHERE id_venta=20', conexion)
assert venta_20.empty
pd.testing.assert_frame_equal(antes, ventas_tras_rollback)
assert conexion.execute("SELECT COUNT(*) FROM lotes WHERE estado='ERROR_TECNICO'").fetchone()[0] == 1
print('Venta 20 después del rollback:')
display(venta_20)
print('Ventas anteriores conservadas:')
display(ventas_tras_rollback)
print('Historial completo de intentos:')
display(pd.read_sql_query(
    'SELECT estado, total, nuevas, existentes FROM lotes ORDER BY rowid', conexion
))


Conflicto esperado: Conflicto de contenido para una venta ya publicada
Venta 20 después del rollback:


,id_venta,id_producto,fecha_venta,monto_centavos,moneda,fila_fuente,lote_inicial


Ventas anteriores conservadas:


,id_venta,id_producto,fecha_venta,monto_centavos,moneda,fila_fuente,lote_inicial
0,1,101,2026-10-05,150000,ARS,v01,3e036ddf70094df482ea85783799ec17
1,7,102,2026-10-05,70000,USD,v08,3e036ddf70094df482ea85783799ec17
2,8,102,2026-09-30,80000,ARS,v09,3e036ddf70094df482ea85783799ec17
3,10,101,2026-10-05,100000,ARS,v11,3e036ddf70094df482ea85783799ec17


Historial completo de intentos:


,estado,total,nuevas,existentes
0,BLOQUEADO,12,0,0
1,PUBLICADO,4,4,0
2,PUBLICADO,4,0,4
3,ERROR_TECNICO,2,0,0


## 15. Cerrar la práctica

**Acción:** cierra la conexión solo después de revisar todas las salidas anteriores.

**Salida:** confirmación del cierre. La base temporal deja de existir; las salidas guardadas en la notebook siguen visibles.

**Para repetir:** reinicia el kernel y comienza por el paso 1.


In [42]:
conexion.close()
print('Conexión cerrada. Práctica completada: bloqueo, carga, reconciliación, idempotencia y rollback.')


Conexión cerrada. Práctica completada: bloqueo, carga, reconciliación, idempotencia y rollback.


## Resultados esperados y extensiones

- **Lote original:** queda bloqueado, no publica ventas y registra los hallazgos.
- **Subconjunto de prueba:** inserta 4 ventas, sin incluir el email en el destino.
- **Recarga idéntica:** informa 0 ventas nuevas y 4 ya existentes; los datos no cambian.
- **Conflicto:** revierte la transacción, deja ausente la venta 20 y preserva las ventas anteriores.

### Ejercicios para discutir o resolver

1. Sustituye `:memory:` por un archivo SQLite y comprueba que la recarga sigue evitando duplicados entre sesiones. Define cómo proteger y eliminar ese archivo.
2. Diseña una cuarentena persistente mínima con lote, fila, regla, estado de revisión y plazo de retención. Evita guardar datos personales en logs generales.
3. Propón categorías separadas para errores de contenido y errores técnicos. ¿Cuáles permitirían un reintento automático y cuáles requieren autorización?
4. Añade una reconciliación por producto y moneda y un control del volumen esperado. Explica cómo una extracción incompleta puede ser válida sintácticamente y aun así no ser apta para publicar.

> Este laboratorio no reemplaza el control de concurrencia distribuido, la autorización institucional ni una política formal para corregir ventas. Una actualización autorizada debe conservar el antes, el después y su justificación.